In [0]:
import requests as req
import json
import time

# 1. Identificar quem ainda falta baixar
todas_frentes = [row['id'] for row in spark.table("bronze_frentes").select("id").collect()]

try:
    # Verifica quem já está na tabela de membros (se ela existir)
    frentes_processadas = [row['id_frente_parlamentar'] for row in spark.table("bronze_membros_frentes").select("id_frente_parlamentar").distinct().collect()]
    frentes_faltantes = [f for f in todas_frentes if f not in frentes_processadas]
except:
    # Se a tabela não existe, falta tudo
    frentes_faltantes = todas_frentes
    frentes_processadas = []

print(f"Total: {len(todas_frentes)} | Já processadas: {len(frentes_processadas)} | Faltam: {len(frentes_faltantes)}")

todos_membros = []
contador = 0

# 2. Loop apenas nas frentes que faltam
for id_frente in frentes_faltantes[:10]:
    url = f"https://dadosabertos.camara.leg.br/api/v2/frentes/{id_frente}/membros"
    
    try:
        # Aumentamos o timeout e adicionamos uma pausa maior
        response = req.get(url, timeout=30)
        
        if response.status_code == 200:
            dados_membros = response.json().get('dados', [])
            for membro in dados_membros:
                membro['id_frente_parlamentar'] = id_frente
                todos_membros.append(membro)
            
            contador += 1
            time.sleep(3.0) # Pausa maior para evitar bloqueio da API
        else:
            print(f"Aviso: Frente {id_frente} retornou status {response.status_code}")

    except Exception as e:
        print(f"Erro na frente {id_frente}: {e}")
        # Se der erro de conexão, salvamos o que temos e paramos para não perder dados
        break 

# 3. Salva os novos dados (APPEND em vez de OVERWRITE)
if todos_membros:
    json_rdd = spark.sparkContext.parallelize([json.dumps(m) for m in todos_membros])
    df_novos_membros = spark.read.json(json_rdd)
    
    # .mode("append") é o segredo para ir somando os dados
    df_novos_membros.write.mode("append").saveAsTable("bronze_membros_frentes")
    print(f"Sucesso: Mais {contador} frentes adicionadas!")
else:
    print("Nada novo para adicionar agora.")

In [0]:
display(spark.table("bronze_membros_frentes"))